In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# 1. 基本設定
# ============================================================

K = 10                 # 腕の数
T = 100                # 最大試行回数
N_RUNS = 5000          # シミュレーション反復回数

INITIAL_BUDGET = 20    # 初期予算。必要に応じて 5, 10, 50 などに変更可

P_GOOD = 0.65          # 良腕の当たり確率
P_BAD = 0.44           # 通常腕の当たり確率

SEED = 12345          #同じ乱数結果を保証

# 学習率 alpha：小・中・大
ALPHA_LEVELS = {
    "low": 0.05,
    "middle": 0.20,
    "high": 0.80
}

# 探索率 epsilon：小・中・大
EPSILON_LEVELS = {
    "low": 0.02,
    "middle": 0.10,
    "high": 0.40
}



In [ ]:
# ============================================================
# 2. 1条件のシミュレーション関数
# ============================================================


#指定された奴は入力不要
def simulate_budget_path(
    alpha,
    epsilon,
    initial_budget=INITIAL_BUDGET,
    K=K,
    T=T,
    n_runs=N_RUNS,
    p_good=P_GOOD,
    p_bad=P_BAD,
    seed=0
):
    """
    指定した alpha, epsilon のもとで、
    予算 W_t の推移をシミュレーションする。

    返り値:
        expected_budget_path: 各時点 t における予算の期待値 E[W_t]
        result_summary: 最終予算・破産率・良腕選択率などの要約
    """

    #特定シードでの乱数生成期
    rng = np.random.default_rng(seed)

    # 各runごとに、良腕の位置をランダムに決める。size分の配列(リスト)を作成する
    good_arms = rng.integers(0, K, size=n_runs)

    # 各run・各腕の価値推定値 Q
    # 各シミュレーションを行に、各腕を列に取った二次元配列で、要素は全て0
    Q = np.zeros((n_runs, K))

    # 各runの予算
    # 全run分の配列を作成し、値をinitial_budgetで埋める
    wealth = np.full(n_runs, float(initial_budget))

    # 予算の時系列記録
    # budget_paths[:, 0] は初期予算
    # budget_paths はシミュレーションを行に、回数を列にしたリスト
    # 全ての行の第１列の値をwealth(初期予算)で上書き
    budget_paths = np.zeros((n_runs, T + 1))
    budget_paths[:, 0] = wealth

    # 指標記録
    # 初めは全て0で埋める。破産の部分はbool(真理値)を見る。0 = Falise, 1 = True
    total_reward = np.zeros(n_runs)
    steps_played = np.zeros(n_runs, dtype=int)
    bankrupt = np.zeros(n_runs, dtype=bool)
    good_arm_count = np.zeros(n_runs)

    # 生存中かどうか
    # 真理値を見る。全て1で埋める
    alive = np.ones(n_runs, dtype=bool)


    #np.where()[0]は()内のTrue部分の位置を返して配列にする
    for t in range(1, T + 1):
        active = np.where(alive)[0]

        # すでに破産したrunは予算0のまま
        # t: はt回以降
        if len(active) == 0:
            budget_paths[:, t:] = 0
            break

        # -----------------------------
        # epsilon-greedy による行動選択
        # -----------------------------

        # greedy選択
        # 初期はQが同点になりやすいので、微小乱数でタイブレーク
        # unioformは範囲での一様分布。sizeの部分は今回は二次元配列をつくる
        jitter = rng.uniform(0, 1e-12, size=(len(active), K))
        greedy_actions = np.argmax(Q[active] + jitter, axis=1)

        # ランダム探索
        random_actions = rng.integers(0, K, size=len(active))

        # epsilonの確率で探索、それ以外はgreedy
        # (真理値, Trueの時, Falseの時)
        explore = rng.random(len(active)) < epsilon
        actions = np.where(explore, random_actions, greedy_actions)

        # -----------------------------
        # 報酬発生
        # -----------------------------

        selected_is_good = actions == good_arms[active]
        hit_prob = np.where(selected_is_good, p_good, p_bad)

        hits = rng.random(len(active)) < hit_prob

        # 当たりなら +1、外れなら -1
        rewards = np.where(hits, 1.0, -1.0)

        # -----------------------------
        # 価値更新
        # Q_{t+1} = Q_t + alpha (R_t - Q_t)
        # -----------------------------

        old_q = Q[active, actions]
        Q[active, actions] = old_q + alpha * (rewards - old_q)

        # -----------------------------
        # 予算更新
        # -----------------------------

        wealth[active] += rewards
        total_reward[active] += rewards
        steps_played[active] += 1
        good_arm_count[active] += selected_is_good

        # 破産判定
        newly_bankrupt = wealth[active] <= 0

        if np.any(newly_bankrupt):
            bankrupt_indices = active[newly_bankrupt]
            bankrupt[bankrupt_indices] = True
            alive[bankrupt_indices] = False
            wealth[bankrupt_indices] = 0.0

        # 予算を記録
        budget_paths[:, t] = wealth

    # 各時点の予算期待値 E[W_t]
    expected_budget_path = budget_paths.mean(axis=0)

    # 要約統計
    safe_steps = np.maximum(steps_played, 1)

    result_summary = {
        "alpha": alpha,
        "epsilon": epsilon,
        "initial_budget": initial_budget,
        "final_expected_budget": expected_budget_path[-1],
        "mean_total_reward": total_reward.mean(),
        "bankruptcy_rate": bankrupt.mean(),
        "mean_steps": steps_played.mean(),
        "good_arm_rate": (good_arm_count / safe_steps).mean()
    }

    return expected_budget_path, result_summary




In [ ]:
# ============================================================
# 3. 9通りの条件を実行
# ============================================================

paths = {}
summaries = []

counter = 0

for alpha_name, alpha_value in ALPHA_LEVELS.items():
    for epsilon_name, epsilon_value in EPSILON_LEVELS.items():
        counter += 1

        print(
            f"Running condition {counter}/9: "
            f"alpha={alpha_name}({alpha_value}), "
            f"epsilon={epsilon_name}({epsilon_value})"
        )

        path, summary = simulate_budget_path(
            alpha=alpha_value,
            epsilon=epsilon_value,
            seed=SEED + counter
        )

        key = (alpha_name, epsilon_name)
        paths[key] = path

        summary["alpha_level"] = alpha_name
        summary["epsilon_level"] = epsilon_name
        summaries.append(summary)

df_summary = pd.DataFrame(summaries)

print("\n=== 9通りの結果要約 ===")
display(df_summary)




In [ ]:
# ============================================================
# 4. 9通りの予算期待値 E[W_t] を図示
# ============================================================

time = np.arange(T + 1)

fig, axes = plt.subplots(
    nrows=3,
    ncols=3,
    figsize=(15, 12),
    sharex=True,
    sharey=True
)

alpha_order = ["low", "middle", "high"]
epsilon_order = ["low", "middle", "high"]

for i, alpha_name in enumerate(alpha_order):
    for j, epsilon_name in enumerate(epsilon_order):
        ax = axes[i, j]

        path = paths[(alpha_name, epsilon_name)]

        alpha_value = ALPHA_LEVELS[alpha_name]
        epsilon_value = EPSILON_LEVELS[epsilon_name]

        ax.plot(time, path)
        ax.axhline(INITIAL_BUDGET, linestyle="--", linewidth=1)

        ax.set_title(
            f"alpha={alpha_name} ({alpha_value}), "
            f"epsilon={epsilon_name} ({epsilon_value})"
        )

        ax.set_xlabel("Time step")
        ax.set_ylabel("Expected budget E[W_t]")
        ax.grid(True)

plt.suptitle(
    f"Expected budget paths under 9 combinations "
    f"(initial budget={INITIAL_BUDGET})",
    fontsize=16
)

plt.tight_layout()
plt.show()




In [ ]:
# ============================================================
# 5. 9通りを1枚の折れ線グラフに重ねる
# ============================================================

plt.figure(figsize=(12, 7))

for alpha_name in alpha_order:
    for epsilon_name in epsilon_order:
        path = paths[(alpha_name, epsilon_name)]
        alpha_value = ALPHA_LEVELS[alpha_name]
        epsilon_value = EPSILON_LEVELS[epsilon_name]

        label = f"alpha={alpha_value}, epsilon={epsilon_value}"
        plt.plot(time, path, label=label)

plt.axhline(INITIAL_BUDGET, linestyle="--", linewidth=1)
plt.xlabel("Time step")
plt.ylabel("Expected budget E[W_t]")
plt.title(f"Expected budget paths: 9 combinations")
plt.legend()
plt.grid(True)
plt.show()




In [ ]:
# ============================================================
# 6. 最終予算期待値のヒートマップ
# ============================================================

pivot_final_budget = df_summary.pivot(
    index="alpha_level",
    columns="epsilon_level",
    values="final_expected_budget"
)

# 表示順を固定
pivot_final_budget = pivot_final_budget.loc[alpha_order, epsilon_order]

plt.figure(figsize=(8, 6))
plt.imshow(pivot_final_budget.values, aspect="auto", origin="lower")
plt.colorbar(label="Final expected budget")

plt.xticks(
    ticks=np.arange(len(epsilon_order)),
    labels=epsilon_order
)

plt.yticks(
    ticks=np.arange(len(alpha_order)),
    labels=alpha_order
)

plt.xlabel("Exploration rate epsilon")
plt.ylabel("Learning rate alpha")
plt.title("Final expected budget by alpha and epsilon")

# 各セルに数値を表示
for i in range(len(alpha_order)):
    for j in range(len(epsilon_order)):
        value = pivot_final_budget.values[i, j]
        plt.text(
            j,
            i,
            f"{value:.2f}",
            ha="center",
            va="center"
        )

plt.show()



In [ ]:

# ============================================================
# 7. 破産率のヒートマップ
# ============================================================

pivot_bankruptcy = df_summary.pivot(
    index="alpha_level",
    columns="epsilon_level",
    values="bankruptcy_rate"
)

pivot_bankruptcy = pivot_bankruptcy.loc[alpha_order, epsilon_order]

plt.figure(figsize=(8, 6))
plt.imshow(pivot_bankruptcy.values, aspect="auto", origin="lower")
plt.colorbar(label="Bankruptcy rate")

plt.xticks(
    ticks=np.arange(len(epsilon_order)),
    labels=epsilon_order
)

plt.yticks(
    ticks=np.arange(len(alpha_order)),
    labels=alpha_order
)

plt.xlabel("Exploration rate epsilon")
plt.ylabel("Learning rate alpha")
plt.title("Bankruptcy rate by alpha and epsilon")

for i in range(len(alpha_order)):
    for j in range(len(epsilon_order)):
        value = pivot_bankruptcy.values[i, j]
        plt.text(
            j,
            i,
            f"{value:.2f}",
            ha="center",
            va="center"
        )

plt.show()



In [ ]:

# ============================================================
# 8. 良腕選択率のヒートマップ
# ============================================================

pivot_good_arm = df_summary.pivot(
    index="alpha_level",
    columns="epsilon_level",
    values="good_arm_rate"
)

pivot_good_arm = pivot_good_arm.loc[alpha_order, epsilon_order]

plt.figure(figsize=(8, 6))
plt.imshow(pivot_good_arm.values, aspect="auto", origin="lower")
plt.colorbar(label="Good arm selection rate")

plt.xticks(
    ticks=np.arange(len(epsilon_order)),
    labels=epsilon_order
)

plt.yticks(
    ticks=np.arange(len(alpha_order)),
    labels=alpha_order
)

plt.xlabel("Exploration rate epsilon")
plt.ylabel("Learning rate alpha")
plt.title("Good arm selection rate by alpha and epsilon")

for i in range(len(alpha_order)):
    for j in range(len(epsilon_order)):
        value = pivot_good_arm.values[i, j]
        plt.text(
            j,
            i,
            f"{value:.2f}",
            ha="center",
            va="center"
        )

plt.show()



In [ ]:

# ============================================================
# 9. 最良条件の表示
# ============================================================

best_row = df_summary.loc[df_summary["final_expected_budget"].idxmax()]

print("\n=== 最終予算期待値が最大の条件 ===")
print(best_row)

print("\nCSVとして保存します。")
df_summary.to_csv("budget_bandit_9_conditions_summary.csv", index=False)
print("Saved: budget_bandit_9_conditions_summary.csv")